# GIT Setup


In [9]:
#RUN TO START SESSION

from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ADL/Project/Product-Search


!git config --global user.name "emdil99"
!git config --global user.email "emdilauro99@gmail.com"

from google.colab import userdata
token = userdata.get('github_key')
assert token is not None, "GitHub token not found"

repo_url = f"https://{token}@github.com/emdil99/Product-Search.git"
!git remote set-url origin $repo_url
!git pull origin main

print("Git remote updated securely using hidden token.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ADL/Project/Product-Search
From https://github.com/emdil99/Product-Search
 * branch            main       -> FETCH_HEAD
Already up to date.
Git remote updated securely using hidden token.


In [8]:
#RUN TO END CODE SESSION


!git add search_engine.ipynb
!git commit -m "Data Structuring"
!git push origin main

print("Updated notebook on GitHub")

[main 87d6145] Data Structuring
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite search_engine.ipynb (82%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 5.12 KiB | 328.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/emdil99/Product-Search.git
   5921ae6..87d6145  main -> main
Updated notebook on GitHub


In [2]:
!pip -q install sentence-transformers faiss-cpu
!pip install -q pyarrow pandas
#!git clone https://github.com/amazon-science/esci-data.git
#!ls esci-data/shopping_queries_dataset



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.5 MB/s eta 0:00:00


# Data Structuring

In [4]:
import pandas as pd

#path = "esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet"

#df = pd.read_parquet(path, filters = [("product_locale", "==", "us")])

#df.columns
#df.head()

#df.to_parquet("eng_products.parquet", index=False)
eng_products = pd.read_parquet("eng_products.parquet")


In [5]:
eng_products.head()



,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [7]:
#from this data, I want to build the embeddings for the search function based off product title to start
#play around with bullet description in a later variation

products = eng_products[["product_id","product_title"]].drop_duplicates().copy()
products = products.rename(columns={"product_title":"product_text"})
products.head()

,product_id,product_text
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...


# Embeddings

In [10]:
from sentence_transformers import SentenceTransformer as ST
model = ST("sentence-transformers/all-mpnet-base-v2")
import numpy as np
from tqdm import tqdm


#sample_texts = products["product_text"].astype(str).head(5).tolist()
#sample_vecs = model.encode(sample_texts, normalize_embeddings=True)

#print("sample_vecs shape:", sample_vecs.shape)
#print("first vector, first 5 numbers:", sample_vecs[0][:5])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sample_vecs shape: (5, 768)
first vector, first 5 numbers: [-0.04092478  0.01422983 -0.02469526  0.00806474 -0.03985495]


In [ ]:
texts = products["product_text"].astype(str).tolist()
batch_size = 256
all_vectos = []


for i in tqdm(range(0,len(texts), batch_size)):
  batch_texts = texts[i:i+batch_size]
  batch_vecs = model.encode(batch_texts, normalize_embeddings=True)
  all_vectors.append(batch_vecs)

product_emb = np.vstack(all_vectors).astype("float32")
